# 하이브리드 추천 시스템 (Hybrid Recommender)

**하이브리드 추천 시스템**은 하나의 추천 방식만 사용하는 것이 아니라, 여러 추천 방식의 결과를 함께 반영하여 더 안정적인 추천을 만드는 방법이다.

1. 콘텐츠 기반 추천
   - 영화의 `genres` 정보를 바탕으로 비슷한 영화를 찾는다.
2. 아이템 기반 협업 필터링
   - 사용자 평점 패턴을 바탕으로 영화 간 유사도를 구한다.

그리고 마지막에는 두 점수를 가중합하여 하이브리드 추천 결과를 만드는 흐름까지 구현해본다.

## 왜 하이브리드 추천이 필요한가

콘텐츠 기반 추천은 영화 자체의 속성 정보를 잘 활용할 수 있다.
예를 들어 장르가 비슷한 영화, 키워드가 비슷한 영화를 찾는 데 적합하다.

하지만 콘텐츠 기반 추천만 사용하면 실제 사용자들의 평점 패턴은 충분히 반영하지 못할 수 있다.

반대로 협업 필터링은 비슷한 취향을 가진 사용자들의 행동 데이터를 활용할 수 있다는 장점이 있다.
하지만 평점 데이터가 적은 새 영화나 활동이 적은 사용자에 대해서는 추천이 약해질 수 있다.

그래서 실제 추천 시스템에서는 한 가지 방식만 고집하기보다 여러 방식을 함께 활용하는 경우가 많다.

In [55]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

In [56]:
# 영화 정보와 평점 정보 로드
movies_df = pd.read_csv('./data/ml-latest-small/movies.csv')
ratings_df = pd.read_csv('./data/ml-latest-small/ratings.csv')

print(movies_df.shape)
print(ratings_df.shape)

movies_df.head()

(9742, 3)
(100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [57]:
# 평점 데이터 간단 확인
ratings_df['rating'].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [58]:
# 영화 정보와 평점 정보 병합
# - 이후 제목(title) 기준으로 평점과 영화 속성을 함께 다루기 쉽도록 만든다.
movie_rating_df = pd.merge(
    ratings_df,
    movies_df,
    on='movieId',
    how='inner'
)

movie_rating_df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


## 1. 콘텐츠 기반 추천

영화의 `genres` 정보만 사용하여
콘텐츠 기반 추천 점수를 만들어본다.

MovieLens의 장르 정보는 `Adventure|Fantasy|Sci-Fi`처럼
`|` 문자로 여러 장르가 연결되어 있다.

이 문자열을 텍스트처럼 다루어 벡터화한 뒤,
코사인 유사도로 영화 간 콘텐츠 유사도를 계산한다.

In [59]:
# genres 문자열을 CountVectorizer에 넣기 쉽게 공백 기준 단어처럼 변환
# 예: 'Adventure|Fantasy|Sci-Fi' -> 'Adventure Fantasy Sci-Fi'
movies_df['genres_literal'] = movies_df['genres'].str.replace('|', ' ', regex=False)

movies_df[['title', 'genres', 'genres_literal']].head()

,title,genres,genres_literal
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


In [60]:
# 장르 텍스트를 숫자 벡터로 변환
count_vect = CountVectorizer(min_df=1, ngram_range=(1, 1))
genre_mat = count_vect.fit_transform(movies_df['genres_literal'])

print(genre_mat.shape)

(9742, 24)


In [61]:
# 영화-영화 콘텐츠 유사도 계산
content_sim = cosine_similarity(genre_mat, genre_mat)
print(content_sim.shape)

# 제목을 index/column으로 지정하여 보기 쉽게 DataFrame으로 변환
content_sim_df = pd.DataFrame(
    content_sim,
    index=movies_df['title'],
    columns=movies_df['title']
)

content_sim_df.head()

(9742, 9742)


title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),1.000000,0.774597,0.316228,0.258199,0.447214,0.0,0.316228,0.632456,0.0,0.258199,...,0.400000,0.316228,0.316228,0.447214,0.0,0.670820,0.774597,0.00000,0.316228,0.447214
Jumanji (1995),0.774597,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.000000,0.000000,0.0,0.288675,0.333333,0.00000,0.000000,0.000000
Grumpier Old Men (1995),0.316228,0.000000,1.000000,0.816497,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.316228,0.000000,0.500000,0.000000,0.0,0.353553,0.408248,0.00000,0.000000,0.707107
Waiting to Exhale (1995),0.258199,0.000000,0.816497,1.000000,0.577350,0.0,0.816497,0.000000,0.0,0.000000,...,0.258199,0.408248,0.816497,0.000000,0.0,0.288675,0.333333,0.57735,0.000000,0.577350
Father of the Bride Part II (1995),0.447214,0.000000,0.707107,0.577350,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,0.447214,0.000000,0.707107,0.000000,0.0,0.500000,0.577350,0.00000,0.000000,1.000000


In [62]:
# 특정 영화와 장르 기준으로 유사한 영화 확인
content_sim_df['Toy Story (1995)'].sort_values(ascending=False)[:10]

title
Toy Story (1995)                                           1.0
Moana (2016)                                               1.0
Adventures of Rocky and Bullwinkle, The (2000)             1.0
The Good Dinosaur (2015)                                   1.0
Antz (1998)                                                1.0
Asterix and the Vikings (Astérix et les Vikings) (2006)    1.0
Emperor's New Groove, The (2000)                           1.0
Toy Story 2 (1999)                                         1.0
Shrek the Third (2007)                                     1.0
Turbo (2013)                                               1.0
Name: Toy Story (1995), dtype: float64

## 2. 아이템 기반 협업 필터링

이번에는 사용자 평점 데이터를 이용하여
영화-영화 유사도를 구해본다.

흐름은 다음과 같다.

1. 사용자-영화 평점 행렬을 만든다.
2. 이를 전치하여 영화-사용자 행렬로 바꾼다.
3. 영화 간 코사인 유사도를 계산한다.
4. 유사한 영화들의 평점을 이용하여 예측 평점을 계산한다.

In [63]:
# 사용자-영화 평점 행렬 생성
# - 행: 사용자
# - 열: 영화
# - 값: 평점
# - 평점이 없는 경우는 0으로 채운다.
user_movie_df = movie_rating_df.pivot_table(
    'rating',
    index='userId',
    columns='title',
    fill_value=0
)

print(user_movie_df.shape)
user_movie_df.head()

(610, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
# 영화-사용자 행렬로 전치
# - 영화-영화 유사도를 구하려면 각 영화가 하나의 벡터가 되도록 바꾸는 것이 직관적이다.
movie_user_df = user_movie_df.T

print(movie_user_df.shape)
movie_user_df.head()

(9719, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [65]:
# 영화-영화 평점 유사도 계산
movie_sim = cosine_similarity(movie_user_df, movie_user_df)
print(movie_sim.shape)

movie_sim_df = pd.DataFrame(
    movie_sim,
    index=user_movie_df.columns,
    columns=user_movie_df.columns
)

movie_sim_df.head()

(9719, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,0.707107,1.000000,0.000000,0.000000,0.0,0.176777,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.857493,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.857493,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


In [66]:
# 특정 영화와 평점 패턴 기준으로 유사한 영화 확인
movie_sim_df['Toy Story (1995)'].sort_values(ascending=False)[:10]

title
Toy Story (1995)                                     1.000000
Toy Story 2 (1999)                                   0.572601
Jurassic Park (1993)                                 0.565637
Independence Day (a.k.a. ID4) (1996)                 0.564262
Star Wars: Episode IV - A New Hope (1977)            0.557388
Forrest Gump (1994)                                  0.547096
Lion King, The (1994)                                0.541145
Star Wars: Episode VI - Return of the Jedi (1983)    0.541089
Mission: Impossible (1996)                           0.538913
Groundhog Day (1993)                                 0.534169
Name: Toy Story (1995), dtype: float64

### 가중 평점 예측을 위한 함수

아이템 기반 협업 필터링에서는
특정 영화와 비슷한 영화들의 평점을 이용하여 예측 평점을 계산할 수 있다.

각 영화마다 유사도가 높은 상위 `topn`개의 영화만 사용하고,
사용자가 실제로 평점을 준 영화만 반영한다.

In [67]:
# 실제 평점과 예측 평점의 MSE 계산 함수
# - 여기서 0은 실제 0점이 아니라 '평점 없음'을 의미하므로
#   실제로 평점이 있는 위치만 골라 평가한다.
def movie_rating_mse(actual, pred):
    non_zero_idx = actual.nonzero()
    actual = actual[non_zero_idx]
    pred = pred[non_zero_idx]
    return mean_squared_error(actual, pred)

In [68]:
# 아이템 기반 협업 필터링 예측 평점 계산
def predict_ratings_item_based_fast(topn=20):
    ratings = user_movie_df.values          # (num_users, num_movies)
    sim = movie_sim_df.values               # (num_movies, num_movies)

    num_users, num_movies = ratings.shape
    pred = np.zeros((num_users, num_movies))

    # 영화 loop
    for movie_idx in range(num_movies):
        # 현재 영화와 유사도가 높은 영화 인덱스를 정렬
        topn_sim_idx = np.argsort(sim[movie_idx])[::-1]

        # 자기 자신 제외 후 상위 topn개만 선택
        topn_sim_idx = topn_sim_idx[topn_sim_idx != movie_idx][:topn]

        # 현재 영화와 topn 유사 영화들의 유사도
        topn_sim = sim[movie_idx, topn_sim_idx]  # (topn,)

        # 모든 사용자의 topn 유사 영화 평점
        topn_ratings = ratings[:, topn_sim_idx]  # (num_users, topn)

        # 사용자가 실제로 평점을 준 영화만 반영
        non_zero_mask = topn_ratings != 0

        # 분자: 유사도 * 평점의 합
        numer = (topn_ratings * topn_sim * non_zero_mask).sum(axis=1)

        # 분모: 실제로 평점이 있는 유사 영화들의 유사도 절댓값 합
        denom = (np.abs(topn_sim) * non_zero_mask).sum(axis=1)

        # 0으로 나누기 방지
        pred[:, movie_idx] = np.divide(
            numer,
            denom,
            out=np.zeros(num_users),
            where=denom != 0
        )

    return pred

ratings_pred_item = predict_ratings_item_based_fast(topn=20)
print(ratings_pred_item.shape)

(610, 9719)


In [69]:
# 예측 평점 DataFrame 변환
rating_pred_item_df = pd.DataFrame(
    ratings_pred_item,
    index=user_movie_df.index,
    columns=user_movie_df.columns
)

rating_pred_item_df.head(1)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,4.523442,4.0


In [70]:
# 아이템 기반 협업 필터링 예측 평점 평가
movie_rating_mse(user_movie_df.values, ratings_pred_item)

1.1087203661474683

## 3. 하이브리드 추천 준비

하이브리드 추천에서는
콘텐츠 기반 점수와 협업 필터링 점수를 함께 사용한다.

다음과 같이 단순한 점수 결합 방식을 사용한다.

하이브리드 점수 = (콘텐츠 기반 점수 × alpha) + (협업 필터링 점수 × beta)

단, 두 점수의 범위가 다를 수 있으므로
먼저 0~1 범위로 정규화한 뒤 결합한다.

In [71]:
# 사용자가 안 본 영화 목록 추출
def get_unseen_movies(user_idx):
    user_ratings = user_movie_df.iloc[user_idx]
    return user_ratings[user_ratings == 0]

In [72]:
# 0 ~ 1 범위로 정규화 하는 함수
def min_max_normalize(series):
    min_val = series.min()
    max_val = series.max()
    
    if max_val == min_val:
        return pd.Series(np.zeros(len(series)), index=series.index)
    
    return (series - min_val) / (max_val - min_val)

In [73]:
# 콘텐츠 기반 점수 계산
# 사용자가 좋아한 영화들과 비슷한 영화일수록 콘텐츠 기반 점수를 높게 준다.
def get_content_based_scores_for_user(user_idx, rating_threshold=4):
    user_ratings = user_movie_df.iloc[user_idx]
    
    # 높게 평가한 영화만 선택
    liked_movies = user_ratings[user_ratings >= rating_threshold].index
    
    # 높게 평가한 영화가 하나도 없으면 모든 영화 점수를 0으로 반환
    if len(liked_movies) == 0:
        return pd.Series(np.zeros(len(content_sim_df.columns)), index=content_sim_df.columns)
    
    # 좋아한 영화들과의 콘텐츠 유사도 평균
    content_scores = content_sim_df.loc[liked_movies].mean(axis=0)
    
    return content_scores

### 콘텐츠 기반 점수와 협업 필터링 점수의 축 맞추기

주의할 점은 두 점수의 영화 목록이 같은 순서와 같은 제목 집합을 가져야 한다는 것이다.

- `content_sim_df`는 `movies_df['title']` 기준
- `rating_pred_item_df`는 `user_movie_df.columns` 기준

MovieLens 데이터에서는 평점이 한 번도 없는 영화가 있을 수 있으므로,
하이브리드 결합 전에는 공통 영화 집합만 사용하도록 맞춘다.

In [74]:
# 두 추천 방식이 모두 점수를 가지고 있는 영화만 골라서 결합
common_titles = rating_pred_item_df.columns.intersection(content_sim_df.columns)

# 공통 영화 집합 기준 콘텐츠 유사도 행렬 정렬
content_sim_common_df = content_sim_df.loc[common_titles, common_titles]

# 아이템 기반 협업 필터링 예측 평점도 같은 열 순서로 정렬
rating_pred_item_common_df = rating_pred_item_df[common_titles]

## 4. 하이브리드 추천 함수 구현

하이브리드 추천 함수는 다음 순서로 동작한다.

1. 사용자가 안 본 영화 목록을 구한다.
2. 해당 사용자에 대한 콘텐츠 기반 점수를 계산한다.
3. 협업 필터링 예측 평점을 가져온다.
4. 두 점수를 각각 정규화한다.
5. 가중합으로 하이브리드 점수를 계산한다.
6. 점수가 높은 순으로 추천한다.

In [75]:
def recommend_movies_hybrid(user_idx, topn=20, alpha=0.4, beta=0.6, rating_threshold=4):
    
    # 사용자가 아직 보지 않은 영화 목록
    unseen = get_unseen_movies(user_idx).index
    
    # 공통 영화 집합 기준 안 본 영화 선택
    unseen = unseen.intersection(common_titles)
    
    # 콘텐츠 기반 점수
    content_scores = get_content_based_scores_for_user(user_idx=user_idx, rating_threshold=rating_threshold)[common_titles]
    
    # 협업 필터링 점수
    cf_scores = rating_pred_item_common_df.iloc[user_idx]
    
    # 안 본 영화만 남기기
    content_scores = content_scores[unseen]
    cf_scores = cf_scores[unseen]
    
    # 점수 정규화
    content_score_norm = min_max_normalize(content_scores)
    cf_scores_norm = min_max_normalize(cf_scores)
    
    # 가중 합으로 하이브리드 점수 계산
    hybrid_scores = alpha * content_score_norm + beta * cf_scores_norm
    
    # 점수가 높은 순으로 정렬 후 상위 topn개 선택
    top_items = hybrid_scores.sort_values(ascending=False)[:topn]
    
    # [확인용] 추천 된 영화들의 실제 사용자 평점 확인
    user_rating_df = user_movie_df.iloc[user_idx][top_items.index]

    return pd.DataFrame({
        'title' : top_items,
        'hybrid_score' : top_items.values,
        'user_rating' : user_rating_df.values,
        'content_score' : content_score_norm[top_items.index].values,
        'cf_score' : cf_scores_norm[top_items.index].values
    })

In [76]:
# 특정 사용자에 대한 하이브리드 추천 결과 확인
recommend_movies_hybrid(user_idx=100, topn=20, alpha=0.4, beta=0.6)

,title,hybrid_score,user_rating,content_score,cf_score
title,,,,,
To Die For (1995),0.994801,0.994801,0.0,0.987003,1.0
Welcome to the Dollhouse (1995),0.983455,0.983455,0.0,0.958638,1.0
Three Colors: White (Trzy kolory: Bialy) (1994),0.983455,0.983455,0.0,0.958638,1.0
Crimes and Misdemeanors (1989),0.966138,0.966138,0.0,0.915346,1.0
Fear and Loathing in Las Vegas (1998),0.956983,0.956983,0.0,0.892457,1.0
"Opposite of Sex, The (1998)",0.946930,0.946930,0.0,0.867326,1.0
Fast Times at Ridgemont High (1982),0.946930,0.946930,0.0,0.867326,1.0
Eat Drink Man Woman (Yin shi nan nu) (1994),0.946930,0.946930,0.0,0.867326,1.0
Léon: The Professional (a.k.a. The Professional) (Léon) (1994),0.941904,0.941904,0.0,0.854759,1.0


## 5. [참고] 아이템 기반 협업 필터링 결과와 비교

하이브리드 추천은
콘텐츠 기반 점수와 협업 필터링 점수를 결합한 결과이다.

아래와 같이 아이템 기반 협업 필터링 단독 추천과 비교해보면
하이브리드 추천이 어떤 방향으로 달라지는지 확인할 수 있다.

In [77]:
# 아이템 기반 협업 필터링 단독 추천 함수
def recommmend_movies_item_based(user_idx, topn=20):
    unseen = get_unseen_movies(user_idx).index
    unseen = pd.Index(unseen).intersection(rating_pred_item_common_df.columns)
    
    temp = rating_pred_item_common_df.iloc[user_idx].sort_values(ascending=False)[:topn]
    user_rating_df = user_movie_df.iloc[user_idx][temp.index]
    
    return pd.DataFrame({
        'title' : temp.index,
        'pred_rating' : temp.values,
        'user_rating' : user_rating_df.values
    })

In [78]:
# 하이브리드 추천과 아이템 기반 협업 필터링 추천 비교
hybrid_titles = set(recommend_movies_hybrid(100, topn=20)['title'])
item_titles = set(recommmend_movies_item_based(100, topn=20)['title'])

print('공통 영화 수:', len(hybrid_titles & item_titles))
print('하이브리드에만 있는 영화 수:', len(hybrid_titles - item_titles))
print('아이템 기반 협업 필터링에만 있는 영화 수:', len(item_titles - hybrid_titles))

공통 영화 수: 0
하이브리드에만 있는 영화 수: 11
아이템 기반 협업 필터링에만 있는 영화 수: 20
